# MarketMind-Q: Classical ML vs QML Finance Benchmark

This notebook summarizes the reproducible benchmark implemented in `src/`. Run the CLI commands in the README first to generate the frozen dataset, metrics, and figures.

## Core Question

Can a quantum kernel classifier identify short-horizon, market-relative ETF upside under low-data and hardware-realistic constraints?

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
dataset_path = ROOT / 'data' / 'marketmind_qml_dataset.csv'
metrics_path = ROOT / 'results' / 'metrics_summary.csv'

dataset = pd.read_csv(dataset_path, parse_dates=['date'])
metrics = pd.read_csv(metrics_path)
dataset.head(), metrics.head()

## Dataset Definition

The target is `1` when a sector ETF beats SPY over the next five trading days by more than 0.25%, and `0` otherwise. Features are past-only lag, volatility, volume, relative-strength, and calendar features.

In [ ]:
dataset.groupby('target').size().rename('rows')

## Top Results

In [ ]:
metrics.sort_values('roc_auc', ascending=False).head(10)[[
    'model', 'model_family', 'execution_mode', 'train_size', 'feature_dim',
    'balanced_accuracy', 'f1', 'roc_auc', 'signal_sharpe',
    'kernel_circuit_depth', 'kernel_two_qubit_gates', 'shots'
]]

## Figures

The report command writes the required judge-facing plots to `results/figures/`.

In [ ]:
from IPython.display import Image, display

for name in [
    'qml_edge_heatmap.png',
    'regime_breakdown.png',
    'confusion_matrices.png',
    'equity_curve.png',
    'score_cost_frontier.png',
]:
    path = ROOT / 'results' / 'figures' / name
    if path.exists():
        display(Image(filename=str(path)))

## qBraid Compiler-Aware Benchmark

The qBraid layer compiles the same quantum-kernel workload through QASM2 roundtrip and Cirq conversion paths, then compares output preservation against compiled resource cost.

In [ ]:
qbraid_metrics_path = ROOT / 'results' / 'qbraid_compile_metrics.csv'
if qbraid_metrics_path.exists():
    qbraid_metrics = pd.read_csv(qbraid_metrics_path)
    display(qbraid_metrics.groupby(['strategy', 'execution_environment'], as_index=False)[[
        'abs_probability_error', 'hellinger_distance', 'depth', 'two_qubit_gates', 'transpile_seconds'
    ]].mean())
else:
    print('Run: python -m src.qbraid_benchmark --config configs/qbraid.yaml')

In [ ]:
for name in ['qbraid_quality_cost.png', 'qbraid_strategy_resources.png']:
    path = ROOT / 'results' / 'figures' / name
    if path.exists():
        display(Image(filename=str(path)))

## Final Claim Template

QML does not universally outperform classical ML. The useful result is the boundary: where quantum kernels are competitive, where they degrade under shot/noise constraints, and what circuit/runtime cost is paid for that performance.

This notebook is for research and education only. It is not investment advice.